# Import packages

In [1]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg
import segmenteverygrain.interactions as si
import os
import rasterio

# %matplotlib qt

2026-03-04 11:36:11.141174: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 11:36:14.788537: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 11:36:33.153045: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Load models

Maybe you need to clone first SAM2. Then add the yaml file in the bottom of the next chunk

In [2]:
# UNET model
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)

# Auto-detect device: CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback
import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"



2026-03-04 11:41:08.001022: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Build in Terrabyte:

In [11]:
import subprocess, textwrap, shutil, os, torch

print("torch cuda:", torch.cuda.is_available())
print("torch device count:", torch.cuda.device_count())

print("nvidia-smi exists:", shutil.which("nvidia-smi"))
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print("nvidia-smi -L:\n", out.stdout or out.stderr)
else:
    print("No nvidia-smi in PATH")

torch cuda: False
torch device count: 0
nvidia-smi exists: None
No nvidia-smi in PATH


In [6]:
from hydra.core.global_hydra import GlobalHydra
GlobalHydra.instance().clear()  # optional, aber in Jupyter oft nötig

from sam2.build_sam import build_sam2

cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"  # NICHT /dss/...
ckpt = "/dss/dsshome1/0B/di54doz/segmenteverygrain/models/sam2.1_hiera_large.pt"

sam = build_sam2(cfg, ckpt, device=device)

AssertionError: GlobalHydra is not initialized, use @hydra.main() or call one of the hydra initialization methods first

Built in private Laptop:

In [ ]:
# # SAM 2.1 checkpoints. Download from:
# # https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

# sam = build_sam2(r"C:\Users\leoni\sam2\sam2\configs\sam2.1\sam2.1_hiera_l.yaml", r"C:\Users\leoni\segmenteverygrain\models\sam2.1_hiera_large.pt", device=device)


## Set your directories

Change here in Terrabyte !

In [ ]:
dir = os.chdir("C:/Users/leoni/kirgis_repo/segmenteverygrain/leonieexamples/2509_with_vegetation_mask_nodataset")
dir = os.getcwd()
os.makedirs("ouputgpkg", exist_ok=True)
os.makedirs("inputtiles", exist_ok=True)
os.makedirs("pebbleplots", exist_ok=True)
os.makedirs("histplots", exist_ok=True)
os.makedirs("csv_stats", exist_ok=True)

inputtiledir = os.path.join(dir, "inputtiles/")
ouputgpkg = os.path.join(dir, "ouputgpkg/")
csvdir = os.path.join(dir, "csv_stats")
plotdirgravel = os.path.join(dir, "pebbleplots/")
plotdirhist = os.path.join(dir, "histplots/")

# Run segmentation

## Not including Masks and saving all statistiks, plots and images and gpkgs

If working not in terrabyte, then adjust your promt number maybe to 2500.

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img
import rasterio

# zusätzliche Imports für Nachbearbeitung (ohne deine Library-Zelle zu ändern)
import geopandas as gpd
from shapely.geometry import Polygon
from skimage.measure import regionprops_table

# --- optional: prompt cap for SAM (None = no limit) ---
MAX_SAM_PROMPTS = 2500
PROMPT_SUBSAMPLE_MODE = "random"   # "random" oder "first"
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")

if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")


# --- loop ---
for i, fname in enumerate(tiles, start=1):
    tile_id = os.path.splitext(os.path.basename(fname))[0]
    print(f"[{i}/{len(tiles)}] {tile_id}")

    # dont calculate nodata tiles (under 100% is allowed)
    with rasterio.open(fname) as src:
        m = src.dataset_mask()
        if not np.any(m):
            print(" -> skipped (100% Nodata)")
            continue

    # load + predict
    image = np.array(load_img(fname))
    image_pred = seg.predict_image(image, unet, I=256)

    # prompts (Unet -> points)
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)

    # --- optional: cap number of prompts before SAM ---
    coords = np.asarray(coords)
    if MAX_SAM_PROMPTS is not None and len(coords) > MAX_SAM_PROMPTS:
        n_before = len(coords)

        if PROMPT_SUBSAMPLE_MODE == "first":
            keep_idx = np.arange(MAX_SAM_PROMPTS)
        else:  # random
            keep_idx = np.sort(rng.choice(len(coords), size=MAX_SAM_PROMPTS, replace=False))

        coords = coords[keep_idx]

        # labels nur mitfiltern, wenn es ein 1D prompt-label array ist
        try:
            labels_arr = np.asarray(labels)
            if labels_arr.ndim == 1 and len(labels_arr) == n_before:
                labels = labels_arr[keep_idx]
        except Exception:
            pass

        print(f"Prompt cap active: reduced prompts from {n_before} -> {len(coords)}")

    # SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam,
        image,
        image_pred,
        coords,
        labels,
        min_area=min_area_px,
        plot_image=True,
        remove_edge_grains=True,
        remove_large_objects=True,
    )

    # --------------------------------------------------
    # 1) Segmentierungsplot speichern (dein bestehender Plot)
    # --------------------------------------------------
    seg_plot_path = os.path.join(plotdirgravel, f"{tile_id}.png")
    fig.savefig(seg_plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved segmentation plot")

    # --------------------------------------------------
    # 2) Polygone georeferenzieren (Pixel -> UTM / CRS des Tiles)
    # --------------------------------------------------
    with rasterio.open(fname) as dataset:
        projected_polys = []
        for grain in all_grains:
            # grain.exterior.xy liefert (x_coords, y_coords) in Pixelkoordinaten
            # hier: row = y, col = x
            px_x = np.asarray(grain.exterior.xy[0])
            px_y = np.asarray(grain.exterior.xy[1])

            x_proj, y_proj = rasterio.transform.xy(
                dataset.transform,
                px_y,   # rows
                px_x    # cols
            )

            poly = Polygon(np.vstack((x_proj, y_proj)).T)
            projected_polys.append(poly)

        # GeoDataFrame erzeugen
        gdf = gpd.GeoDataFrame({"geometry": projected_polys}, geometry="geometry", crs=dataset.crs)

        # --------------------------------------------------
        # 3) Eigenschaften / Statistiken aus labeled image
        # --------------------------------------------------
        # (intensity_image hier weggelassen, da nur geometrische Properties gebraucht werden)
        props = regionprops_table(
            labels.astype("int"),
            properties=("label", "area", "centroid", "major_axis_length", "minor_axis_length"),
        )
        grain_stats = pd.DataFrame(props)

        # Falls aus irgendeinem Grund Anzahl nicht exakt passt, robust behandeln
        if len(grain_stats) != len(gdf):
            nmin = min(len(grain_stats), len(gdf))
            print(f"Warning: len(gdf)={len(gdf)} != len(grain_stats)={len(grain_stats)} -> truncating to {nmin}")
            gdf = gdf.iloc[:nmin].copy()
            grain_stats = grain_stats.iloc[:nmin].copy()

        # Zentroiden (row,col) -> CRS coords
        centroid_x, centroid_y = rasterio.transform.xy(
            dataset.transform,
            grain_stats["centroid-0"].values,  # rows
            grain_stats["centroid-1"].values,  # cols
        )

        # Pixelgröße aus Transform (X-Richtung); für quadratische Pixel ok
        px_size_x = abs(dataset.transform.a)

        # Attribute in GDF schreiben
        gdf["label"] = grain_stats["label"].values
        gdf["area_px"] = grain_stats["area"].values
        gdf["centroid_x"] = centroid_x
        gdf["centroid_y"] = centroid_y
        gdf["major_axis_px"] = grain_stats["major_axis_length"].values
        gdf["minor_axis_px"] = grain_stats["minor_axis_length"].values
        gdf["major_axis_length"] = grain_stats["major_axis_length"].values * px_size_x
        gdf["minor_axis_length"] = grain_stats["minor_axis_length"].values * px_size_x
        gdf["major_axis_mm"] = gdf["major_axis_length"] * 1000.0
        gdf["minor_axis_mm"] = gdf["minor_axis_length"] * 1000.0

    # --------------------------------------------------
    # 4) Histogramm-Plot speichern (NICHT den Segmentierungsplot überschreiben)
    # --------------------------------------------------
    if len(gdf) > 0:
        fig_hist, ax_hist = seg.plot_histogram_of_axis_lengths(
            gdf["major_axis_mm"],
            gdf["minor_axis_mm"],
            binsize=0.25,
            xlimits=[8, 2 * 256],
        )
        hist_plot_path = os.path.join(plotdirhist, f"{tile_id}_hist.png")  # eigener Name!
        fig_hist.savefig(hist_plot_path, dpi=200, bbox_inches="tight")
        plt.close(fig_hist)
        print("Saved histogram plot")
    else:
        print("No grains found -> skipping histogram plot")

    # --------------------------------------------------
    # 5) GPKG + Statistik-CSV speichern
    # --------------------------------------------------
    gpkg_path = os.path.join(ouputgpkg, f"{tile_id}_grains.gpkg")
    csv_path = os.path.join(csvdir, f"{tile_id}_grain_stats.csv")

    # GPKG
    gdf.to_file(gpkg_path, driver="GPKG")
    print(f"Saved GPKG: {gpkg_path}")

    # CSV (ohne Geometrie; Geometrie steckt im GPKG)
    stats_df = gdf.drop(columns="geometry").copy()
    stats_df.to_csv(csv_path, index=False)
    print(f"Saved stats CSV: {csv_path}")

    print("done with segmentation + export")

Found 7 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/7] tile_r000006_c000024_masked
segmenting image tiles...


100%|██████████| 3/3 [00:02<00:00,  1.12it/s]


creating masks using SAM...


100%|██████████| 726/726 [01:20<00:00,  8.99it/s]


finding overlapping polygons...


...

finding best polygons...


100%|██████████| 71/71 [00:01<00:00, 43.06it/s]


creating labeled image...


100%|██████████| 104/104 [00:00<00:00, 231.29it/s]


Saved segmentation plot
Saved histogram plot
Saved GPKG: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\ouputgpkg/tile_r000008_c000032_masked_grains.gpkg
Saved stats CSV: C:\Users\leoni\kirgis_repo\segmenteverygrain\leonieexamples\2509_with_vegetation_mask_nodataset\csv_stats\tile_r000008_c000032_masked_grain_stats.csv
done with segmentation + export
